# MjWarp Tutorial
- Simple model & data
- visualize
- Step & Forward

### 1. Mujoco model & data
- simple foor & panda

In [1]:
# 1-1. import libraries
import mujoco
import mujoco.viewer
import mujoco_warp as mjw

import os
import sys
import time
import glfw
import numpy as np
import warp as wp

# 1-2. import pp base mujoco
sys.path.append(os.path.abspath('./'))
from pp_base_mujoco.UTILS import *
from pp_base_mujoco.VIEWER import MUJOCOGLVIEWER

In [2]:
# 1-3. get model & data
xml_path = './asset/panda_scene.xml'
xml_abs_path = os.path.abspath(xml_path)
model = mujoco.MjModel.from_xml_path(xml_abs_path)
data = mujoco.MjData(model)

In [3]:
joint_names = get_joint_names(model, data)
print("joint_names:", joint_names)
initial_qpos = np.array([0, -0.5, 0, -2.5, 0, 2.0, 0])
apply_qpos_names(model, data, joint_names, initial_qpos)
mujoco.mj_forward(model, data)

joint_names: ['joint1', 'joint2', 'joint3', 'joint4', 'joint5', 'joint6', 'joint7']


In [14]:
viewer = MUJOCOGLVIEWER(model, data, window_size = (3000, 2000))
# viewer.add_cameras(camera_names=["cam_1"], types=["fixed"], sizes=[(1000, 1000)])
# print("fixed camera index:", viewer.fixed_idx)
# viewer.add_cameras(camera_names=["cam_2"], types=["fixed"], sizes=[(1000, 1000)])
# print("fixed camera index:", viewer.fixed_idx)
viewer.add_cameras(camera_names=["cam_1", "cam_2"], types=["fixed", "fixed"], sizes=[(1000, 1000), (1000, 1000)])

while viewer.is_alive():
    mujoco.mj_forward(model, data)
    viewer.render()

viewer.close()
del(viewer)

### 2. MjWarp model & data / Step

In [4]:
world_column = 4
world_row = 2
n_world = world_column * world_row
max_contact_per_world = 50
mjw_model = mjw.put_model(model)
mjw_data = mjw.put_data(model, data, nworld=n_world, nconmax=max_contact_per_world)

Warp 1.12.0.dev20260126 initialized:
   CUDA Toolkit 12.9, Driver 12.4
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 3060" (12 GiB, sm_86, mempool enabled)
   Kernel cache:
     /home/juju/.cache/warp/1.12.0.dev20260126


In [5]:
# shape
print("mujoco qpos shape:", data.qpos.shape)
print("mjwarp qpos shape:", mjw_data.qpos.shape)

# type
print("mujoco qpos type:", data.qpos.dtype)
print("mjwarp qpos type:", mjw_data.qpos.dtype)

mujoco qpos shape: (7,)
mjwarp qpos shape: (8, 7)
mujoco qpos type: float64
mjwarp qpos type: <class 'warp._src.types.float32'>


In [7]:
# del(viewer)
viewer = MUJOCOGLVIEWER(model, data, window_size = (3000, 2000))

while viewer.is_alive():
  mjw.step(mjw_model, mjw_data)
  # fetch the first world back onto CPU for rendering
  mjw.get_data_into(data, model, mjw_data)
  viewer.render()

viewer.close()

Module mujoco_warp._src.smooth 64ad9e5 load on device 'cuda:0' took 3.67 ms  (cached)
Module mujoco_warp._src.collision_driver de6dc98 load on device 'cuda:0' took 0.21 ms  (cached)
Module _nxn_broadphase__locals__kernel_36c68517 36c6851 load on device 'cuda:0' took 0.21 ms  (cached)
Module ccd_kernel_builder__locals__ccd_kernel_275b2300 275b230 load on device 'cuda:0' took 0.40 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_30157060 b2e79f4 load on device 'cuda:0' took 0.34 ms  (cached)
Module mujoco_warp._src.constraint 1ddad5e load on device 'cuda:0' took 0.37 ms  (cached)
Module _actuator_velocity__locals__actuator_velocity_cba63b3d 9928a50 load on device 'cuda:0' took 0.23 ms  (cached)
Module mujoco_warp._src.passive 0a145a0 load on device 'cuda:0' took 0.32 ms  (cached)
Module mujoco_warp._src.forward 3875c24 load on device 'cuda:0' took 0.29 ms  (cached)
Module mujoco_warp._src.support b7a7a89 load on device 'cuda:0' took 0.20 ms  (cached)
Module _tile

In [6]:
# with wp.ScopedCapture() as capture:
#   mjw.step(model, data)

# mjw.reset_data(model, data)
# frames = []
# for _ in range(3 * 60):
#   wp.capture_launch(capture.graph)

### 3. Batch rendering
- Render context required
- Multiple ways to render warp context
    - simply check scene with mediapy
    - Use GL to make render window

동일한 render context 사용 -> gl에서 이 context 가지고 Render 가능할듯!!
1. context 외에 필요한 것
- window (gl window)
- 

In [6]:
# from render_util import get_rgb
from mujoco_warp._src.types import RenderContext

@wp.kernel
def unpack_rgb_kernel(
  # In:
  packed: wp.array2d(dtype=wp.uint32),
  rgb_adr: wp.array(dtype=int),
  camera_index: int,
  # Out:
  rgb_out: wp.array3d(dtype=wp.vec3),
):
  """Unpack ABGR uint32 packed pixel data into separate R, G, and B channels."""
  worldid, pixelid = wp.tid()

  xid = pixelid % rgb_out.shape[2]
  yid = pixelid // rgb_out.shape[2]

  rgb_adr_offset = rgb_adr[camera_index]
  val = packed[worldid, rgb_adr_offset + pixelid]
  b = wp.float32(val & wp.uint32(0xFF)) * wp.static(1.0 / 255.0)
  g = wp.float32((val >> wp.uint32(8)) & wp.uint32(0xFF)) * wp.static(1.0 / 255.0)
  r = wp.float32((val >> wp.uint32(16)) & wp.uint32(0xFF)) * wp.static(1.0 / 255.0)
  rgb_out[worldid, yid, xid] = wp.vec3(r, g, b)

def get_rgb(rc: RenderContext, camera_index: int, rgb_out: wp.array3d(dtype=wp.vec3)):
  """Get the RGB data output from the render context buffers for a given camera index.

  Args:
    rc: The render context on device.
    camera_index: The index of the camera to get the RGB data for.
    rgb_out: The output array to store the RGB data in, with shape (nworld, height, width).
  """
  wp.launch(
    unpack_rgb_kernel,
    dim=(rgb_out.shape[0], rgb_out.shape[1] * rgb_out.shape[2]),
    inputs=[rc.rgb_data, rc.rgb_adr, camera_index],
    outputs=[rgb_out],
  )

In [7]:
camera_index = 1
camera_resolution = (500, 500)
rc = mjw.create_render_context(
 model,
 nworld=n_world,
 cam_res=camera_resolution,
 render_rgb=True,
 render_depth=True,
)

# Populate data fields for the current state
mjw.forward(mjw_model, mjw_data)
mjw.refit_bvh(mjw_model, mjw_data, rc)

# Render the current state
mjw.render(mjw_model, mjw_data, rc)

rgb_data = wp.zeros((n_world, camera_resolution[1], camera_resolution[0]), dtype=wp.vec3)
get_rgb(rc, camera_index=camera_index, rgb_out=rgb_data)

rgb_grid = rgb_data.numpy().reshape(world_row, world_column, camera_resolution[1], camera_resolution[0], 3) # 3
rgb_grid = rgb_grid.transpose(0, 2, 1, 3, 4)
rgb_grid = rgb_grid.reshape(world_row * camera_resolution[0], world_column * camera_resolution[1], 3)


Module mujoco_warp._src.render_util 815a8d5 load on device 'cuda:0' took 0.31 ms  (cached)
Module mujoco_warp._src.io db47b00 load on device 'cuda:0' took 0.48 ms  (cached)
Module mujoco_warp._src.bvh 3505354 load on device 'cuda:0' took 0.24 ms  (cached)
Module mujoco_warp._src.smooth 64ad9e5 load on device 'cuda:0' took 3.92 ms  (cached)
Module mujoco_warp._src.collision_driver de6dc98 load on device 'cuda:0' took 0.37 ms  (cached)
Module _nxn_broadphase__locals__kernel_36c68517 36c6851 load on device 'cuda:0' took 0.21 ms  (cached)
Module ccd_kernel_builder__locals__ccd_kernel_275b2300 275b230 load on device 'cuda:0' took 0.46 ms  (cached)
Module _primitive_narrowphase__locals__primitive_narrowphase_30157060 b2e79f4 load on device 'cuda:0' took 0.39 ms  (cached)
Module mujoco_warp._src.constraint 1ddad5e load on device 'cuda:0' took 0.41 ms  (cached)
Module _actuator_velocity__locals__actuator_velocity_cba63b3d 9928a50 load on device 'cuda:0' took 0.48 ms  (cached)
Module mujoco_war

In [8]:
from OpenGL.GL import *
import numpy as np
import glfw
import sys

# --- 1. INITIALIZATION & SETUP ---
if not glfw.init():
    print("Failed to initialize GLFW")
    sys.exit()

alpha = 0.7
gl_height = int(world_row * camera_resolution[1])
gl_width = int(world_column * camera_resolution[0])

window = glfw.create_window(gl_width, gl_height, "MuJoCo Warp Live", None, None)
if not window:
    glfw.terminate()
    print("Failed to create GLFW window")
    sys.exit()

glfw.make_context_current(window)

# Create and bind the texture ID
texture = glGenTextures(1)
glBindTexture(GL_TEXTURE_2D, texture)
glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
glPixelStorei(GL_UNPACK_ALIGNMENT, 4)

# ALLOCATE GPU MEMORY ONCE (Passing 'None' creates an empty texture of the right size)
glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, gl_width, gl_height, 0, GL_RGBA, GL_UNSIGNED_BYTE, None)

# Set up blending and background color once
glClearColor(0.2, 0.2, 0.2, 1.0) 
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_TEXTURE_2D)

# Pre-calculate alpha channel to save CPU time inside the loop
alpha_value = int(alpha * 255) 
alpha_channel = np.full((gl_height, gl_width, 1), alpha_value, dtype=np.uint8)

print("Starting render loop. Close the window to exit.")

# --- 2. THE MAIN LOOP ---
try:
    while not glfw.window_should_close(window):
        
        # [!] UPDATE YOUR DATA HERE [!]
        # Fetch your newest rgb_grid from MuJoCo here.
        # For example: rgb_grid = get_new_mujoco_frame()
        
        # Format the new frame
        pixels_uint8_rgb = (rgb_grid * 255).astype(np.uint8)
        pixels_rgba = np.concatenate((pixels_uint8_rgb, alpha_channel), axis=2)
        pixels_rgba = np.ascontiguousarray(pixels_rgba, dtype=np.uint8)
        
        # UPDATE TEXTURE DATA (Fast: overwrites existing memory)
        glBindTexture(GL_TEXTURE_2D, texture)
        glTexSubImage2D(GL_TEXTURE_2D, 0, 0, 0, gl_width, gl_height, GL_RGBA, GL_UNSIGNED_BYTE, pixels_rgba)

        # Clear screen
        glClear(GL_COLOR_BUFFER_BIT)

        # Draw the Quad
        glBegin(GL_QUADS)
        glTexCoord2f(0.0, 1.0); glVertex2f(-1.0, -1.0)
        glTexCoord2f(1.0, 1.0); glVertex2f( 1.0, -1.0)
        glTexCoord2f(1.0, 0.0); glVertex2f( 1.0,  1.0)
        glTexCoord2f(0.0, 0.0); glVertex2f(-1.0,  1.0)
        glEnd()

        # Swap front and back buffers
        glfw.swap_buffers(window)
        
        # Poll for and process events (like window closing, mouse clicks, etc.)
        glfw.poll_events()

except KeyboardInterrupt:
    # Allows you to safely stop the loop in Jupyter by pressing "Interrupt Kernel"
    print("Render loop interrupted by user.")

# --- 3. CLEANUP ---
# Always properly destroy the window when done, especially in Jupyter!
glfw.destroy_window(window)
glfw.terminate()
print("GLFW terminated safely.")

Starting render loop. Close the window to exit.
GLFW terminated safely.


### Real viewer

In [11]:
camera_index = -1
camera_resolution = (500, 500)
start_rc = time.time()
rc = mjw.create_render_context(
 model,
 nworld=n_world,
 cam_res=camera_resolution,
 render_rgb=True,
 render_depth=True,
)
rc_creation_duration = time.time() - start_rc
print(f"Render context creation duration: {rc_creation_duration:.4f} seconds")
rgb_data = wp.zeros((n_world, camera_resolution[1], camera_resolution[0]), dtype=wp.vec3)

def mjwarp_render_rgb(mjw_model, mjw_data, rc, camera_index, camera_resolution, world_grid = (4, 2)):
    world_row, world_column = world_grid
    # Populate data fields for the current state
    mjw.refit_bvh(mjw_model, mjw_data, rc)
    mjw.render(mjw_model, mjw_data, rc)
    get_rgb(rc, camera_index=camera_index, rgb_out=rgb_data)
    # check 
    rgb_grid = rgb_data.numpy()
    print("RGB data shape:", rgb_grid.shape)
    mjw_nworld, col, row, rgb = rgb_grid.shape
    if mjw_nworld * col * row * rgb != world_row * world_column * camera_resolution[0] * camera_resolution[1]*3:
        raise ValueError(f"Unexpected RGB data shape: {rgb_grid.shape}, expected total size: {world_row * world_column * camera_resolution[0] * camera_resolution[1]}")
    else:
        rgb_grid = rgb_data.numpy().reshape(world_row, world_column, camera_resolution[1], camera_resolution[0], 3) # 3
        rgb_grid = rgb_grid.transpose(0, 2, 1, 3, 4)
        rgb_grid = rgb_grid.reshape(world_row * camera_resolution[0], world_column * camera_resolution[1], 3)
        return rgb_grid

# --- 1. INITIALIZATION & SETUP ---
if not glfw.init():
    print("Failed to initialize GLFW")
    sys.exit()

alpha = 0.7
gl_height = int(world_row * camera_resolution[1])
gl_width = int(world_column * camera_resolution[0])

window = glfw.create_window(gl_width, gl_height, "MuJoCo Warp Live", None, None)
if not window:
    glfw.terminate()
    print("Failed to create GLFW window")
    sys.exit()

glfw.make_context_current(window)

# Create and bind the texture ID
texture = glGenTextures(1)
glBindTexture(GL_TEXTURE_2D, texture)
glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
glPixelStorei(GL_UNPACK_ALIGNMENT, 4)

# ALLOCATE GPU MEMORY ONCE (Passing 'None' creates an empty texture of the right size)
glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, gl_width, gl_height, 0, GL_RGBA, GL_UNSIGNED_BYTE, None)

# Set up blending and background color once
glClearColor(0.2, 0.2, 0.2, 1.0) 
glEnable(GL_BLEND)
glBlendFunc(GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA)
glEnable(GL_TEXTURE_2D)

# Pre-calculate alpha channel to save CPU time inside the loop
alpha_value = int(alpha * 255) 
alpha_channel = np.full((gl_height, gl_width, 1), alpha_value, dtype=np.uint8)

print("Starting render loop. Close the window to exit.")

# --- 2. THE MAIN LOOP ---
try:
    while not glfw.window_should_close(window):

        # get glfw window size & update camera resolution 
        gl_width_curr, gl_height_curr = glfw.get_framebuffer_size(window)
        # if gl width and height changed
        if gl_width_curr != gl_width or gl_height_curr != gl_height:
            gl_width, gl_height = gl_width_curr, gl_height_curr
            camera_resolution = (int(gl_width_curr/world_column), int(gl_height_curr/world_row))
            # update render context 
            rc = mjw.create_render_context(
                model,
                nworld=n_world,
                cam_res=camera_resolution,
                render_rgb=True,
                render_depth=True,
                )

        # [!] UPDATE YOUR DATA HERE [!]
        start_time = time.time()
        tick = 10
        for _ in range(tick):
            mjw.step(mjw_model, mjw_data)
        duration = time.time() - start_time
        print("==================================================================")
        print(f"mjw.step duration: {duration:.4f} seconds")
        rgb_grid = mjwarp_render_rgb(mjw_model, mjw_data, rc, camera_index, camera_resolution, world_grid=(world_row, world_column))
        render_duration = time.time() - start_time
        print(f"Render and RGB fetch duration: {render_duration:.4f} seconds")
        # this function is so slow
        
        # Format the new frame
        pixels_uint8_rgb = (rgb_grid * 255).astype(np.uint8)
        uint_change_duration = time.time() - start_time
        print(f"RGB uint8 conversion duration: {uint_change_duration:.4f} seconds")
        pixels_rgba = np.concatenate((pixels_uint8_rgb, alpha_channel), axis=2)
        alpha_concat_duration = time.time() - start_time
        print(f"Alpha channel concatenation duration: {alpha_concat_duration:.4f} seconds")
        pixels_rgba = np.ascontiguousarray(pixels_rgba, dtype=np.uint8)
        pixel_calculation_duration = time.time() - start_time
        print(f"Pixel formatting duration: {pixel_calculation_duration:.4f} seconds") # this takes really long time
        
        # UPDATE TEXTURE DATA (Fast: overwrites existing memory)
        glBindTexture(GL_TEXTURE_2D, texture)
        glTexSubImage2D(GL_TEXTURE_2D, 0, 0, 0, gl_width, gl_height, GL_RGBA, GL_UNSIGNED_BYTE, pixels_rgba)
        gl_texture_duration = time.time() - start_time

        # Clear screen
        glClear(GL_COLOR_BUFFER_BIT)

        # Draw the Quad
        glBegin(GL_QUADS)
        glTexCoord2f(0.0, 1.0); glVertex2f(-1.0, -1.0)
        glTexCoord2f(1.0, 1.0); glVertex2f( 1.0, -1.0)
        glTexCoord2f(1.0, 0.0); glVertex2f( 1.0,  1.0)
        glTexCoord2f(0.0, 0.0); glVertex2f(-1.0,  1.0)
        glEnd()
        gl_draw_duration = time.time() - start_time

        # Swap front and back buffers
        glfw.swap_buffers(window)
        
        # Poll for and process events (like window closing, mouse clicks, etc.)
        glfw.poll_events()

        total_duration = time.time() - start_time
        print(f"Total loop duration: {total_duration:.4f} seconds")

except KeyboardInterrupt:
    # Allows you to safely stop the loop in Jupyter by pressing "Interrupt Kernel"
    print("Render loop interrupted by user.")

# --- 3. CLEANUP ---
# Always properly destroy the window when done, especially in Jupyter!
glfw.destroy_window(window)
glfw.terminate()
print("GLFW terminated safely.")

Render context creation duration: 0.1269 seconds
Starting render loop. Close the window to exit.
mjw.step duration: 0.1054 seconds
RGB data shape: (8, 500, 500, 3)
Render and RGB fetch duration: 0.2002 seconds
RGB uint8 conversion duration: 0.2039 seconds
Alpha channel concatenation duration: 0.2158 seconds
Pixel formatting duration: 0.2158 seconds
Total loop duration: 0.2184 seconds
mjw.step duration: 0.0587 seconds
RGB data shape: (8, 500, 500, 3)
Render and RGB fetch duration: 0.1446 seconds
RGB uint8 conversion duration: 0.1496 seconds
Alpha channel concatenation duration: 0.1614 seconds
Pixel formatting duration: 0.1614 seconds
Total loop duration: 0.1629 seconds
mjw.step duration: 0.0598 seconds
RGB data shape: (8, 500, 500, 3)
Render and RGB fetch duration: 0.1462 seconds
RGB uint8 conversion duration: 0.1507 seconds
Alpha channel concatenation duration: 0.1624 seconds
Pixel formatting duration: 0.1624 seconds
Total loop duration: 0.1639 seconds
mjw.step duration: 0.0585 seconds